# LongFlow — chunked engine A/B (the product-config test)

Runtime: **L4 ok**, ~40–60 min, ~$3. No training. Pre-registration: NOTES
"CHUNKED ENGINE A/B" (2026-08-19).

The golden-window insight: every production chunk restarts from the clean
prompt, so the whole render lives in the first-30-seconds regime Josh
graded 85–90%. This renders the same 803-word script the production way —
4 turn-boundary chunks, fresh generate call each, 0.25 s crossfade stitch —
once per engine. **EMA noise is retired; all polish uses independent
noise.**

Stitched wavs land in `Drive/longflow_chunked/` AND as `40_ENGINE_*` copies
in the TOP PICKS folder. Josh ears them against `ce_teacher`.

| tag | engine |
|---|---|
| ce_teacher | stock DDPM head (ceiling for this harness) |
| ce_july | July head + CFG heun8, no polish (Josh's current #1) |
| ce_july_p2 | July head + audio polish k=2 (independent noise) |
| ce_cleanabl_p2 | cleanabl + both-polish k=2 (the #1-tie recipe) |
| ce_cleanabl | cleanabl plain (control) |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Chunked engine A/B v1.0 (2026-08-19)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
OUT = "/content/chunked"
DRIVE_OUT = "/content/drive/MyDrive/longflow_chunked"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

if os.path.exists(f"{DRIVE_OUT}/chunked_report.json"):
    with open(f"{DRIVE_OUT}/chunked_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/chunked_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

# copy checkpoints to LOCAL disk before torch.load — a Drive-FUSE flake
# mid-read ("Transport endpoint is not connected") killed a run 2026-08-19
LOCAL_CKPT = "/content/ckpts"
os.makedirs(LOCAL_CKPT, exist_ok=True)
for name in ("full10k_20k.pt", "cleanabl_20k_step20000.pt"):
    if not os.path.exists(f"{LOCAL_CKPT}/{name}"):
        shutil.copy(f"{CKPT_DIR}/{name}", f"{LOCAL_CKPT}/{name}")
head_july, mean_j, std_j = load_checkpoint(f"{LOCAL_CKPT}/full10k_20k.pt")
head_july = head_july.to("cuda")
head_cl, mean_c, std_c = load_checkpoint(f"{LOCAL_CKPT}/cleanabl_20k_step20000.pt")
head_cl = head_cl.to("cuda")
print("READY")


In [ ]:
# ===== Script -> 4 turn-boundary chunks (production shape) + polish patch =====
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript_turns(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return turns

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
TURNS = turnscript_turns(ABL_WORDS)  # same script recipe as GN5-8 / all closed loops

# production shape: whole-turn chunks of ~200 words each
CHUNKS, cur, cw = [], [], 0
for t in TURNS:
    cur.append(t); cw += len(t.split())
    if cw >= 200:
        CHUNKS.append("\n".join(cur) + "\n"); cur, cw = [], 0
if cur:
    CHUNKS.append("\n".join(cur) + "\n")
print(f"{sum(len(t.split()) for t in TURNS)} words -> {len(CHUNKS)} chunks "
      f"of ~{[sum(len(l.split()) for l in c.splitlines()) for c in CHUNKS]} words")
report["chunks"] = CHUNKS
report["script_words"] = sum(len(t.split()) for t in TURNS)

def polish(z, cond, neg, k, cfg_scale=1.3, total_steps=10):
    """Independent noise ONLY (EMA retired 2026-08-19 — audible ~1 Hz wobble)."""
    if k <= 0:
        return z
    sched = model.model.noise_scheduler
    sched.set_timesteps(total_steps)
    ts = sched.timesteps[-k:]
    zt = z.to("cuda", torch.bfloat16)
    cond2 = torch.cat([cond, neg], dim=0).to("cuda", torch.bfloat16)
    noise = torch.randn(zt.shape, device="cuda", dtype=torch.float32).to(torch.bfloat16)
    zt = sched.add_noise(zt, noise, ts[0].expand(zt.shape[0]))
    for t in ts:
        combined = torch.cat([zt, zt], dim=0)
        eps = model.model.prediction_head(
            combined, t.repeat(combined.shape[0]).to(combined), condition=cond2)
        c_eps, u_eps = torch.split(eps, len(eps) // 2, dim=0)
        guided = u_eps + cfg_scale * (c_eps - u_eps)
        zt = sched.step(guided, t, zt).prev_sample
    return zt.float()

class ChunkPolishPatch(CFGFlowHeadPatch):
    """CFG flow head + optional polish. audio_k polishes what is heard;
    feed_k what re-enters the loop; equal values = one polished latent for
    both (the 30_MILESTONE recipe when audio_k=feed_k=2)."""

    def __init__(self, *args, audio_k=0, feed_k=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.audio_k = audio_k
        self.feed_k = feed_k
        self.audio_latents = []

    def __enter__(self):
        super().__enter__()
        inner = self.model.sample_speech_tokens
        patch = self

        def flow_sample_pol(condition, neg_condition=None, cfg_scale=None):
            z = inner(condition, neg_condition=neg_condition, cfg_scale=cfg_scale)
            zf = z.float()
            c = condition.float().cuda()
            ng = neg_condition.float().cuda() if neg_condition is not None else None
            if ng is None or (patch.audio_k == 0 and patch.feed_k == 0):
                patch.audio_latents.append(zf.detach().cpu())
                return z
            if patch.audio_k == patch.feed_k:
                zp = polish(zf, c, ng, patch.audio_k)
                patch.audio_latents.append(zp.detach().cpu())
                return zp.to(condition.dtype)
            z_audio = polish(zf, c, ng, patch.audio_k) if patch.audio_k > 0 else zf
            z_feed = polish(zf, c, ng, patch.feed_k) if patch.feed_k > 0 else zf
            patch.audio_latents.append(z_audio.detach().cpu())
            return z_feed.to(condition.dtype)

        self.model.sample_speech_tokens = flow_sample_pol
        return self

def decode_latents_xfade(z, chunk_frames=225, overlap_frames=4):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    step = chunk_frames - overlap_frames
    pieces, shape_fn = [], None
    for i in range(0, z.shape[0], step):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt failed: {repr(e)[:120]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed")
        pieces.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
        if i + chunk_frames >= z.shape[0]:
            break
    if len(pieces) == 1:
        return pieces[0]
    spf = len(pieces[0]) // chunk_frames
    ov = overlap_frames * spf
    outw = pieces[0]
    for p in pieces[1:]:
        fade = np.linspace(0, 1, ov, dtype=np.float32)
        outw[-ov:] = outw[-ov:] * (1 - fade) + p[:ov] * fade
        outw = np.concatenate([outw, p[ov:]])
    return outw

def crossfade_stitch(wavs, sr=24000, fade_s=0.25):
    """Production-style stitch: 0.25 s linear crossfade between chunks."""
    n = int(sr * fade_s)
    out = wavs[0]
    for wv in wavs[1:]:
        if len(out) < n or len(wv) < n:
            out = np.concatenate([out, wv])
            continue
        fade = np.linspace(0, 1, n, dtype=np.float32)
        out[-n:] = out[-n:] * (1 - fade) + wv[:n] * fade
        out = np.concatenate([out, wv[n:]])
    return out


In [ ]:
# ===== Render every engine, chunk by chunk, stitch, save =====
ENGINES = [  # (tag, kind, head/mean/std or None, audio_k, feed_k)
    ("ce_teacher", "teacher", None, 0, 0),
    ("ce_july", "student", (head_july, mean_j, std_j), 0, 0),
    ("ce_july_p2", "student", (head_july, mean_j, std_j), 2, 0),
    ("ce_cleanabl_p2", "student", (head_cl, mean_c, std_c), 2, 2),
    ("ce_cleanabl", "student", (head_cl, mean_c, std_c), 0, 0),
]
for tag, kind, hms, ak, fk in ENGINES:
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    t0 = time.time()
    chunk_wavs = []
    for ci, chunk_text in enumerate(CHUNKS):
        torch.manual_seed(ci)  # per-chunk seed, same across engines
        if kind == "teacher":
            with torch.inference_mode():
                gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                     tokenizer=processor.tokenizer,
                                     cfg_scale=1.3, max_new_tokens=1500)
            wv = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
        else:
            h, mn, sd = hms
            with ChunkPolishPatch(model, h, mn, sd, nfe=8, sway=0.0,
                                  sampler=heun_sample, audio_k=ak,
                                  feed_k=fk) as patch, torch.inference_mode():
                gen = model.generate(**gen_inputs([chunk_text], [[P0]]),
                                     tokenizer=processor.tokenizer,
                                     cfg_scale=1.3, max_new_tokens=1500)
            if ak != fk:
                wv = decode_latents_xfade(torch.cat(patch.audio_latents))
            else:
                wv = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
        chunk_wavs.append(wv)
        print(f"  {tag} chunk {ci+1}/{len(CHUNKS)}: {len(wv)/24000:.1f}s", flush=True)
    stitched = crossfade_stitch(chunk_wavs)
    save_wav(tag, stitched, {"kind": kind, "audio_k": ak, "feed_k": fk,
                             "n_chunks": len(CHUNKS),
                             "wall_s": round(time.time() - t0, 1)})
print("ALL ENGINES RENDERED — stitched wavs in Drive/longflow_chunked/")


## Done — listen

Five stitched ~4–5 min renders in `Drive/longflow_chunked/`. Listen order:
`ce_teacher` first (the ceiling in this harness), then `ce_july`,
`ce_july_p2`, `ce_cleanabl_p2`, `ce_cleanabl`. The question, per the
pre-registration: **does any student engine hold your ~85% grade for the
FULL render now that every chunk restarts in the golden window — and are
the seams inaudible?** Claude will copy the winner into TOP PICKS.
